### JSON Parsing And Processing 

In [1]:
import json
import os
os.makedirs("data/json_files", exist_ok=True)

In [4]:
# Sample nested JSON data

json_data = [
    {
        "company": "TechCorp",

        "employees": [
            {
                "id": 1,
                "name": "John Doe",
                "role": "Software Engineer",
                "skills": ["Python", "JavaScript", "React"],

                "projects": [
                    {"name": "RAG System", "status": "In Progress"},
                    {"name": "Data Pipeline", "status": "Completed"}
                ]
            },

            {
                "id": 2,
                "name": "Jane Smith",
                "role": "Data Scientist",
                "skills": ["Python", "Machine Learning", "SQL"],

                "projects": [
                    {"name": "ML Model", "status": "In Progress"},
                    {"name": "Analytics Dashboard", "status": "Planning"}
                ]
            }
        ],

        "departments": {
            "engineering": {
                "head": "Mike Johnson",
                "budget": 1000000,
                "team_size": 25
            },

            "data_science": {
                "head": "Sarah Williams",
                "budget": 750000,
                "team_size": 15
            }
        }
    }
]

In [5]:
json_data

[{'company': 'TechCorp',
  'employees': [{'id': 1,
    'name': 'John Doe',
    'role': 'Software Engineer',
    'skills': ['Python', 'JavaScript', 'React'],
    'projects': [{'name': 'RAG System', 'status': 'In Progress'},
     {'name': 'Data Pipeline', 'status': 'Completed'}]},
   {'id': 2,
    'name': 'Jane Smith',
    'role': 'Data Scientist',
    'skills': ['Python', 'Machine Learning', 'SQL'],
    'projects': [{'name': 'ML Model', 'status': 'In Progress'},
     {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
  'departments': {'engineering': {'head': 'Mike Johnson',
    'budget': 1000000,
    'team_size': 25},
   'data_science': {'head': 'Sarah Williams',
    'budget': 750000,
    'team_size': 15}}}]

In [6]:
with open("data/json_files/company_data.json", "w") as f:
    json.dump(json_data, f, indent=2)

In [7]:
import json

# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},

    {"timestamp": "2024-01-01", "event": "page_view",
     "user_id": 123, "page": "/home"},

    {"timestamp": "2024-01-01", "event": "purchase",
     "user_id": 123, "amount": 99.99}
]

with open("data/json_files/events.jsonl", "w") as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

## JSON Processsing Strategies

In [12]:
from langchain_community.document_loaders import JSONLoader
import json

# Method 1: JSONLoader with jq_schema

print("JSONLoader - Extract specific fields")

# Extract employee information
employee_loader = JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema=".[].employees[]",   # Correct jq query
    text_content=False
)

employee_docs = employee_loader.load()

print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee document: {employee_docs[0].page_content[:200]}...")

JSONLoader - Extract specific fields
Loaded 2 employee documents
First employee document: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...


In [13]:
# Method 2: Custom JSON processing for complex structures
print("\nCustom JSON processing")

from typing import List
from langchain_core.documents import Document
import json


def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation."""

    with open(filepath, 'r') as f:
        data = json.load(f)

    documents = []

    # data is a list -> take first dictionary
    for emp in data[0].get("employees", []):

        content = f"""
        Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}

        Projects:
        """

        # Add projects
        for proj in emp.get('projects', []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"

        # Create document
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )

        documents.append(doc)

    return documents


Custom JSON processing


In [14]:
process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='\n        Employee Profile:\n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n\n        Projects:\n        \n- RAG System (Status: In Progress)\n- Data Pipeline (Status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='\n        Employee Profile:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n\n        Projects:\n        \n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)')]

In [1]:
from typing import List
from langchain_core.documents import Document
import json


def process_event_json(filepath: str) -> List[Document]:

    # Step 1: Read file
    with open(filepath, "r") as f:
        lines = f.readlines()

    # Step 2: Create empty document list
    documents = []

    # Step 3: Loop through each line
    for line in lines:

        # Convert JSON string into dictionary
        event_data = json.loads(line)

        # Step 4: Create content
        content = f"""
        Event Details:

        Timestamp: {event_data.get('timestamp')}
        Event: {event_data.get('event')}
        User ID: {event_data.get('user_id')}
        """

        # Optional fields
        if "page" in event_data:
            content += f"\nPage: {event_data['page']}"

        if "amount" in event_data:
            content += f"\nAmount: {event_data['amount']}"

        # Step 5: Create document with metadata
        doc = Document(
            page_content=content,

            metadata={
                "source": filepath,
                "event_type": event_data.get("event"),
                "user_id": event_data.get("user_id"),
                "timestamp": event_data.get("timestamp")
            }
        )

        # Step 6: Add document into list
        documents.append(doc)

    # Step 7: Return documents
    return documents


# Function call
docs = process_event_json("data/json_files/events.jsonl")

# Print documents
for doc in docs:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)


        Event Details:

        Timestamp: 2024-01-01
        Event: user_login
        User ID: 123
        
{'source': 'data/json_files/events.jsonl', 'event_type': 'user_login', 'user_id': 123, 'timestamp': '2024-01-01'}
--------------------------------------------------

        Event Details:

        Timestamp: 2024-01-01
        Event: page_view
        User ID: 123
        
Page: /home
{'source': 'data/json_files/events.jsonl', 'event_type': 'page_view', 'user_id': 123, 'timestamp': '2024-01-01'}
--------------------------------------------------

        Event Details:

        Timestamp: 2024-01-01
        Event: purchase
        User ID: 123
        
Amount: 99.99
{'source': 'data/json_files/events.jsonl', 'event_type': 'purchase', 'user_id': 123, 'timestamp': '2024-01-01'}
--------------------------------------------------
